In [1]:
import duckdb
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

conn = duckdb.connect("../data/mbta.duckdb", read_only=True)

def query(sql: str) -> pl.DataFrame:
    return conn.execute(sql).pl()

# Check data volume
counts = query("""
    SELECT 
        (SELECT count(*) FROM intermediate.int_scheduled_vs_actual) as predictions,
        (SELECT count(*) FROM intermediate.int_scheduled_vs_actual WHERE delay_seconds IS NOT NULL) as with_delay,
        (SELECT count(DISTINCT route_id) FROM intermediate.int_scheduled_vs_actual) as routes,
        (SELECT count(DISTINCT stop_id) FROM intermediate.int_scheduled_vs_actual) as stops,
        (SELECT min(extracted_at) FROM intermediate.int_scheduled_vs_actual) as earliest,
        (SELECT max(extracted_at) FROM intermediate.int_scheduled_vs_actual) as latest
""")
print("Data Coverage:")
print(counts)

Data Coverage:
shape: (1, 6)
┌─────────────┬────────────┬────────┬───────┬─────────────────┬────────────────────────────┐
│ predictions ┆ with_delay ┆ routes ┆ stops ┆ earliest        ┆ latest                     │
│ ---         ┆ ---        ┆ ---    ┆ ---   ┆ ---             ┆ ---                        │
│ i64         ┆ i64        ┆ i64    ┆ i64   ┆ datetime[μs]    ┆ datetime[μs]               │
╞═════════════╪════════════╪════════╪═══════╪═════════════════╪════════════════════════════╡
│ 8723        ┆ 7986       ┆ 7      ┆ 254   ┆ 2026-04-19      ┆ 2026-04-19 16:05:45.044212 │
│             ┆            ┆        ┆       ┆ 15:11:41.943332 ┆                            │
└─────────────┴────────────┴────────┴───────┴─────────────────┴────────────────────────────┘


In [2]:
delays = query("""
    SELECT delay_seconds, delay_category, route_name, route_type_desc
    FROM intermediate.int_scheduled_vs_actual
    WHERE delay_seconds IS NOT NULL
      AND delay_seconds BETWEEN -600 AND 1200
""").to_pandas()

print(f"Total observations with delay data: {len(delays)}")
print("\nDelay statistics (seconds):")
print(f"  Mean:   {delays['delay_seconds'].mean():.1f}")
print(f"  Median: {delays['delay_seconds'].median():.1f}")
print(f"  Std:    {delays['delay_seconds'].std():.1f}")
print(f"  Min:    {delays['delay_seconds'].min():.1f}")
print(f"  Max:    {delays['delay_seconds'].max():.1f}")

fig = px.histogram(
    delays,
    x="delay_seconds",
    nbins=80,
    title="Distribution of Delays (seconds)",
    labels={"delay_seconds": "Delay (seconds)"},
    color_discrete_sequence=["#636EFA"],
)
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="On time")
fig.add_vline(x=60, line_dash="dash", line_color="orange", annotation_text="1 min late")
fig.add_vline(x=300, line_dash="dash", line_color="red", annotation_text="5 min late")
fig.update_layout(xaxis_title="Delay (seconds)", yaxis_title="Count")
fig.show()

Total observations with delay data: 7924

Delay statistics (seconds):
  Mean:   -31.9
  Median: -34.0
  Std:    302.7
  Min:    -592.0
  Max:    1044.0


In [3]:
category_counts = query("""
    SELECT 
        delay_category,
        count(*) as count,
        round(100.0 * count(*) / sum(count(*)) over(), 2) as pct
    FROM intermediate.int_scheduled_vs_actual
    WHERE delay_seconds IS NOT NULL
    GROUP BY delay_category
    ORDER BY 
        CASE delay_category
            WHEN 'early' THEN 1
            WHEN 'on_time' THEN 2
            WHEN 'slightly_late' THEN 3
            WHEN 'late' THEN 4
            WHEN 'very_late' THEN 5
        END
""").to_pandas()

print("Delay Category Distribution:")
print(category_counts)

fig = px.bar(
    category_counts,
    x="delay_category",
    y="pct",
    text="pct",
    title="Delay Category Distribution (%)",
    color="delay_category",
    color_discrete_map={
        "early": "#2ecc71",
        "on_time": "#27ae60",
        "slightly_late": "#f39c12",
        "late": "#e74c3c",
        "very_late": "#c0392b",
    },
)
fig.update_layout(showlegend=False, yaxis_title="Percentage")
fig.show()

Delay Category Distribution:
  delay_category  count    pct
0          early   3587  44.92
1        on_time   2253  28.21
2  slightly_late   1128  14.12
3           late    729   9.13
4      very_late    289   3.62


In [4]:
route_delays = query("""
    SELECT * FROM marts.mart_route_reliability
    ORDER BY reliability_score DESC
""").to_pandas()

print("Route Reliability Metrics:")
print(route_delays[['route_name', 'route_type_desc', 'total_predictions', 'on_time_pct',
                      'avg_delay_seconds', 'median_delay_seconds', 'p90_delay_seconds',
                      'reliability_score']].to_string(index=False))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("On-Time Percentage by Route", "Average Delay by Route (seconds)")
)

fig.add_trace(
    go.Bar(
        x=route_delays["route_name"],
        y=route_delays["on_time_pct"],
        marker_color=route_delays["reliability_score"].apply(
            lambda x: "#2ecc71" if x > 80 else "#f39c12" if x > 50 else "#e74c3c"
        ),
        text=route_delays["on_time_pct"].apply(lambda x: f"{x}%"),
        textposition="outside",
        name="On-Time %",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=route_delays["route_name"],
        y=route_delays["avg_delay_seconds"],
        marker_color=route_delays["avg_delay_seconds"].apply(
            lambda x: "#e74c3c" if x > 60 else "#f39c12" if x > 0 else "#2ecc71"
        ),
        text=route_delays["avg_delay_seconds"].apply(lambda x: f"{x:.0f}s"),
        textposition="outside",
        name="Avg Delay",
    ),
    row=1, col=2,
)

fig.update_layout(height=500, title_text="Route Performance Comparison", showlegend=False)
fig.show()

Route Reliability Metrics:
  route_name route_type_desc  total_predictions  on_time_pct  avg_delay_seconds  median_delay_seconds  p90_delay_seconds  reliability_score
    Red Line      Heavy Rail                394        90.61              -55.4                 -59.0               60.0               93.5
 Orange Line      Heavy Rail                335        99.10             -252.4                -363.0               21.0               91.1
Green Line E      Light Rail                271        91.88             -275.6                -331.0                6.0               83.4
Green Line B      Light Rail                189        66.67               -6.5                  -4.0              319.6               64.9
Green Line C      Light Rail                144        39.58              153.7                 186.5              400.5               34.7
Green Line D      Light Rail                144        18.75              241.1                 230.0              724.4             

In [5]:
delays_by_route = query("""
    SELECT delay_seconds, route_name, route_type_desc
    FROM intermediate.int_scheduled_vs_actual
    WHERE delay_seconds IS NOT NULL
      AND delay_seconds BETWEEN -600 AND 1200
""").to_pandas()

fig = px.box(
    delays_by_route,
    x="route_name",
    y="delay_seconds",
    color="route_type_desc",
    title="Delay Distribution by Route",
    labels={"delay_seconds": "Delay (seconds)", "route_name": "Route"},
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_hline(y=300, line_dash="dash", line_color="red", annotation_text="5 min threshold")
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

In [6]:
hotspots = query("""
    SELECT 
        stop_name, municipality, latitude, longitude,
        avg_delay_seconds, median_delay_seconds, late_pct,
        total_predictions, routes_served, delay_hotspot_score
    FROM marts.mart_stop_performance
    WHERE total_predictions >= 3
      AND latitude IS NOT NULL
    ORDER BY delay_hotspot_score DESC
""").to_pandas()

print(f"Stops with delay data: {len(hotspots)}")
print("\nTop 10 Delay Hotspots:")
print(hotspots[['stop_name', 'municipality', 'avg_delay_seconds', 'late_pct', 'delay_hotspot_score']].head(10).to_string(index=False))

fig = px.scatter_map(
    hotspots,
    lat="latitude",
    lon="longitude",
    size="delay_hotspot_score",
    color="avg_delay_seconds",
    hover_name="stop_name",
    hover_data=["municipality", "late_pct", "total_predictions"],
    color_continuous_scale="RdYlGn_r",
    title="Delay Hotspot Map (larger = more delays)",
    zoom=11,
    height=600,
    size_max=20,
)
fig.update_layout(map_style="carto-positron")
fig.show()

Stops with delay data: 201

Top 10 Delay Hotspots:
        stop_name municipality  avg_delay_seconds  late_pct  delay_hotspot_score
    Suffolk Downs       Boston              625.7     100.0                100.0
         Maverick       Boston              582.8     100.0                100.0
       Wonderland       Revere              668.9     100.0                100.0
   Orient Heights       Boston              649.7     100.0                100.0
            State       Boston              582.8     100.0                100.0
        Beachmont       Revere              694.9     100.0                100.0
     Revere Beach       Revere              683.9     100.0                100.0
Government Center       Boston              564.8     100.0                100.0
         Aquarium       Boston              600.8     100.0                100.0
          Bowdoin       Boston              545.0     100.0                100.0


In [7]:
alerts = query("""
    SELECT 
        effect, severity_category,
        count(*) as alert_count,
        round(avg(impact_score), 2) as avg_impact,
        round(avg(informed_entity_count), 1) as avg_entities_affected,
        sum(CASE WHEN is_active THEN 1 ELSE 0 END) as active_count
    FROM marts.mart_alert_summary
    GROUP BY effect, severity_category
    ORDER BY avg_impact DESC
""").to_pandas()

print("Alert Impact by Effect Type:")
print(alerts.to_string(index=False))

fig = px.treemap(
    alerts,
    path=["severity_category", "effect"],
    values="alert_count",
    color="avg_impact",
    color_continuous_scale="Reds",
    title="Alert Distribution: Severity → Effect Type (color = impact score)",
)
fig.update_layout(height=500)
fig.show()

Alert Impact by Effect Type:
           effect severity_category  alert_count  avg_impact  avg_entities_affected active_count
       SUSPENSION          critical            8       51.62                   16.9            3
           DETOUR          critical           20       38.96                    8.3            4
          SHUTTLE          critical            8       35.07                   51.6            0
     TRACK_CHANGE          critical            1       31.04                   30.0            0
     STOP_CLOSURE             major            2       27.33                    5.0            2
       BIKE_ISSUE             minor            1       24.91                 1482.0            1
  STATION_CLOSURE          critical            2       24.04                   10.5            0
  PARKING_CLOSURE          critical            1       22.33                    5.0            0
 ELEVATOR_CLOSURE          critical            2       20.88                    4.0            2
 

In [8]:
weather_delays = query("""
    SELECT 
        weather_condition,
        count(*) as predictions,
        round(avg(delay_seconds), 1) as avg_delay,
        round(avg(CASE WHEN delay_seconds > 60 THEN 1.0 ELSE 0.0 END) * 100, 2) as late_pct
    FROM intermediate.int_weather_transit
    WHERE delay_seconds IS NOT NULL
    GROUP BY weather_condition
    ORDER BY avg_delay DESC
""").to_pandas()

print("Delay by Weather Condition:")
print(weather_delays.to_string(index=False))

if len(weather_delays) > 1:
    fig = px.bar(
        weather_delays,
        x="weather_condition",
        y="avg_delay",
        color="late_pct",
        text="predictions",
        title="Average Delay by Weather Condition",
        color_continuous_scale="RdYlGn_r",
    )
    fig.update_layout(xaxis_title="Weather", yaxis_title="Avg Delay (seconds)")
    fig.show()
else:
    print("Only one weather condition in current data — need more days for meaningful comparison")

Delay by Weather Condition:
weather_condition  predictions  avg_delay  late_pct
             Rain         7540      -30.3     28.26
           Cloudy          446     -162.2      3.36


In [9]:
print("=" * 60)
print("DELAY ANALYSIS SUMMARY")
print("=" * 60)

total = query("SELECT count(*) FROM intermediate.int_scheduled_vs_actual WHERE delay_seconds IS NOT NULL").to_pandas().iloc[0, 0]
late = query("SELECT count(*) FROM intermediate.int_scheduled_vs_actual WHERE delay_seconds > 60").to_pandas().iloc[0, 0]

print(f"\nTotal predictions with delay data: {total}")
print(f"Late (>1 min): {late} ({100*late/total:.1f}%)")
print(f"Routes analyzed: {len(route_delays)}")
print(f"Stops analyzed: {len(hotspots)}")

print("""
LIMITATIONS:
- Single extraction snapshot — delay patterns will be more meaningful 
  after accumulating data over multiple days/weeks
- Negative delays (early arrivals) may reflect schedule vs prediction 
  temporal misalignment for the current service day
- Weather correlation requires multi-day data with varying conditions
- Blue Line not appearing may indicate no active predictions at extraction time

NEXT STEPS:
- Accumulate 7+ days of data via scheduler
- Notebook 03: Feature engineering for delay prediction model
- Notebook 04: Train delay prediction models
- Notebook 05: Model evaluation and comparison
""")

conn.close()

DELAY ANALYSIS SUMMARY

Total predictions with delay data: 7986
Late (>1 min): 2146 (26.9%)
Routes analyzed: 7
Stops analyzed: 201

LIMITATIONS:
- Single extraction snapshot — delay patterns will be more meaningful 
  after accumulating data over multiple days/weeks
- Negative delays (early arrivals) may reflect schedule vs prediction 
  temporal misalignment for the current service day
- Weather correlation requires multi-day data with varying conditions
- Blue Line not appearing may indicate no active predictions at extraction time

NEXT STEPS:
- Accumulate 7+ days of data via scheduler
- Notebook 03: Feature engineering for delay prediction model
- Notebook 04: Train delay prediction models
- Notebook 05: Model evaluation and comparison

